In [ ]:
# ============================================================================
# METAME DATA RISK ASSESSMENT SYSTEM - GOOGLE COLAB
# ============================================================================

# STEP 1: Install required libraries (if needed)
!pip install openpyxl -q

import openpyxl
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple

print("Libraries loaded successfully!\n")
print("Please upload Book4.xlsx using the file browser on the left")
print("   (Click the folder icon → Upload icon → Select Book4.xlsx)")
print("\nOnce uploaded, run the rest of the code below.\n")

# ============================================================================
# SECTION 1: EXCEL DATA LOADING AND LOOKUP
# ============================================================================

def load_risk_matrix_from_book4(file_path: str = 'Book4.xlsx', sheet_name: str = 'Sheet8') -> pd.DataFrame:
    wb = openpyxl.load_workbook(file_path, data_only=True)
    sheet = wb[sheet_name]

    # Extract headers starting from column C (skip A and B)
    headers = []
    for col in range(3, sheet.max_column + 1):
        header = sheet.cell(row=1, column=col).value
        if header:
            header_str = str(header).strip()
            # Skip non-dimension headers
            if header_str and header_str not in ['Unnamed', 'Record Factor', '']:
                headers.append(header_str)

    # Extract data types and their risk ratings
    data = []
    for row in range(2, sheet.max_row + 1):
        data_type = sheet.cell(row=row, column=2).value  # Column B
        if data_type:
            row_data = {'Data_Type': str(data_type).strip()}
            for col_idx, header in enumerate(headers, start=3):
                cell_value = sheet.cell(row=row, column=col_idx).value
                row_data[header] = str(cell_value).strip() if cell_value else 'Low'
            data.append(row_data)

    df = pd.DataFrame(data)
    df.set_index('Data_Type', inplace=True)

    return df


Libraries loaded successfully!

Please upload Book4.xlsx using the file browser on the left
   (Click the folder icon → Upload icon → Select Book4.xlsx)

Once uploaded, run the rest of the code below.



In [ ]:
# ============================================================================
# SECTION 2: RISK SCORING CONFIGURATION
# ============================================================================

# Score mapping for qualitative risk levels
SCORE_MAPPING = {
    'High': 3,
    'Medium': 2,
    'Low': 1
}

# Risk dimension weights (based on metaMe framework)
def derive_weights_from_book4(risk_matrix: pd.DataFrame) -> Dict[str, float]:
    """
    Derive dimension weights empirically from Book4 risk matrix.

    METHODOLOGY:
    Dimensions that are frequently rated "High" or "Medium" across many data
    types are more important overall and receive higher weights.

    CALCULATION:
    1. For each dimension, count occurrences of High/Medium/Low across all 105 data types
    2. Calculate frequency score: (High_count × 3) + (Medium_count × 2) + (Low_count × 1)
    3. Normalize scores to create weights that average to 1.0

    JUSTIFICATION:
    - Data-driven: Based on actual risk assessments in Book4
    - Captures expert consensus: Book4 created by 4 domain experts
    - Reflects applicability: Dimensions rarely rated "High" get lower weights

    Args:
        risk_matrix: DataFrame with data types and risk ratings

    Returns:
        Dictionary of dimension weights
    """
    dimension_scores = {}

    for dimension in risk_matrix.columns:
        # Count rating frequencies for this dimension
        high_count = (risk_matrix[dimension] == 'High').sum()
        medium_count = (risk_matrix[dimension] == 'Medium').sum()
        low_count = (risk_matrix[dimension] == 'Low').sum()

        # Calculate weighted frequency score
        frequency_score = (high_count * 3) + (medium_count * 2) + (low_count * 1)
        dimension_scores[dimension] = frequency_score

    # Calculate mean score for normalization
    mean_score = sum(dimension_scores.values()) / len(dimension_scores)

    # Convert to weights with mean of 1.0
    weights = {}
    for dimension, score in dimension_scores.items():
        weights[dimension] = score / mean_score

    return weights


def print_weight_derivation_details(risk_matrix: pd.DataFrame, weights: Dict[str, float]):
    """
    Print detailed breakdown of how weights were derived from Book4.
    """
    print("\n" + "="*70)
    print("EMPIRICAL WEIGHT DERIVATION FROM BOOK4")
    print("="*70)
    print("\nMethodology: Weights based on frequency distribution of risk ratings")
    print("across all 105 data types in Book4.xlsx\n")

    # Create detailed breakdown DataFrame
    breakdown_data = []
    for dimension in risk_matrix.columns:
        high_count = (risk_matrix[dimension] == 'High').sum()
        medium_count = (risk_matrix[dimension] == 'Medium').sum()
        low_count = (risk_matrix[dimension] == 'Low').sum()
        total = len(risk_matrix)

        breakdown_data.append({
            'Dimension': dimension,
            'High Count': high_count,
            'High %': f"{high_count/total*100:.1f}%",
            'Medium Count': medium_count,
            'Med %': f"{medium_count/total*100:.1f}%",
            'Low Count': low_count,
            'Weight': f"{weights[dimension]:.2f}"
        })

    breakdown_df = pd.DataFrame(breakdown_data)
    breakdown_df = breakdown_df.sort_values('Weight', ascending=False, key=lambda x: x.str.replace('Weight', '').astype(float))

    print("Top 10 Highest Weighted Dimensions:")
    print(breakdown_df.head(10).to_string(index=False))

    print("\n" + "="*70 + "\n")

In [ ]:
# ============================================================================
# SECTION 2B: INITIALIZE RISK WEIGHTS (AUTOMATICALLY DERIVED)
# ============================================================================

# NOTE: RISK_WEIGHTS will be calculated automatically when Book4 is loaded

RISK_WEIGHTS = None  # Will be set in main()

In [ ]:
# ============================================================================
# SECTION 3: iQUBE DEFINITIONS
# ============================================================================

IQUBES_DICT = {
    "Open Bank Account Qube": [
        'Name', 'Address', 'DOB', 'Proof of ID', 'Proof of Address',
        'Social Security No', 'Telephone Number', 'Email Address'
    ],
    "New Credit Card Qube": [
        'Name', 'Address', 'DOB', 'Proof of ID', 'Social Security No',
        'Telephone Number', 'Email Address', 'Employment Status', 'Credit Score'
    ],
    "Mortgage Application Qube": [
        'Name', 'Address', 'DOB', 'Proof of ID', 'Proof of Address',
        'Social Security No', 'Telephone Number', 'Email Address',
        'Employment Status', 'Credit Score', 'Income', 'Bank Statements'
    ],
    "Car Finance Qube": [
        'Name', 'Address', 'DOB', 'Proof of ID', 'Social Security No',
        'Telephone Number', 'Email Address', 'Employment Status',
        'Credit Score', 'Income', 'Price of Car'
    ],
    "Student Loan Application Qube": [
        'Name', 'Address', 'DOB', 'Proof of ID', 'Social Security No',
        'Telephone Number', 'Email Address', 'Personal School Fees', 'Course Duration'
    ],
    "Investment Qube": [
        'Name', 'Email Address', 'Employment Status', 'Credit Score',
        'Income', 'Investment Account Details', 'Investments'
    ],
    "Retirement Plan Qube": [
        'Name', 'Address', 'DOB', 'Telephone Number', 'Social Security No',
        'Income', 'Tax Record', 'Health Score', '401(k)', 'IRA (Individual Retirement Account)'
    ],
    "Debt Management Qube": [
        'Name', 'Address', 'DOB', 'Telephone Number', 'Social Security No',
        'Income', 'Credit Score', 'Outstanding Debt', 'Credit Inquiries'
    ]
}


In [ ]:
def calculate_basic_risk_score(
    selected_qubes: List[str],
    risk_matrix: pd.DataFrame
) -> Tuple[int, float, str]:
    """
    Calculate basic (unweighted) risk score for selected iQubes.
    """
    total_score = 0
    max_score = 0

    for qube in selected_qubes:
        if qube not in IQUBES_DICT:
            print(f"Warning: Qube '{qube}' not found in definitions.")
            continue

        for data_type in IQUBES_DICT[qube]:
            if data_type not in risk_matrix.index:
                print(f"Warning: Data type '{data_type}' not found in risk matrix.")
                continue

            # Handle potential duplicate index entries
            try:
                data_row = risk_matrix.loc[data_type]

                # If multiple rows match, take first one
                if isinstance(data_row, pd.DataFrame):
                    data_row = data_row.iloc[0]

                for dimension in risk_matrix.columns:
                    risk_level = data_row[dimension]
                    score = SCORE_MAPPING.get(risk_level, 1)
                    total_score += score
                    max_score += 3
            except Exception as e:
                print(f"Error processing '{data_type}': {e}")
                continue

    percentage_score = (total_score / max_score * 100) if max_score > 0 else 0
    classification = classify_risk(percentage_score)

    return total_score, percentage_score, classification


In [ ]:
# ============================================================================
# SECTION 5: WEIGHTED RISK SCORING
# ============================================================================

def calculate_weighted_risk_score(
    selected_qubes: List[str],
    risk_matrix: pd.DataFrame,
    risk_weights: Dict[str, float]
) -> Tuple[float, float, str]:
    """
    Calculate weighted risk score for selected iQubes.
    """
    # Convert risk matrix to numeric scores
    risk_df = risk_matrix.replace(SCORE_MAPPING).infer_objects(copy=False)

    # Apply weights to each dimension
    weighted_scores = pd.DataFrame(index=risk_df.index)
    for dimension in risk_df.columns:
        weight = risk_weights.get(dimension, 1.0)
        weighted_scores[dimension] = risk_df[dimension].astype(float) * weight

    total_weighted_score = 0
    max_weighted_score = 0

    for qube in selected_qubes:
        if qube not in IQUBES_DICT:
            print(f"Warning: Qube '{qube}' not found in definitions.")
            continue

        for data_type in IQUBES_DICT[qube]:
            if data_type not in weighted_scores.index:
                print(f"Warning: Data type '{data_type}' not found in risk matrix.")
                continue

            # Handle potential duplicate index entries
            try:
                data_row = weighted_scores.loc[data_type]

                # If multiple rows match (DataFrame returned), take first one
                if isinstance(data_row, pd.DataFrame):
                    data_row = data_row.iloc[0]

                # Sum weighted scores for this data type across all dimensions
                total_weighted_score += float(data_row.sum())

                # Calculate max possible weighted score
                for dimension in risk_df.columns:
                    weight = risk_weights.get(dimension, 1.0)
                    max_weighted_score += 3 * weight
            except Exception as e:
                print(f"Error processing '{data_type}': {e}")
                continue

    percentage_score = (total_weighted_score / max_weighted_score * 100) if max_weighted_score > 0 else 0
    classification = classify_risk(percentage_score)

    return total_weighted_score, percentage_score, classification


In [ ]:
# ============================================================================
# SECTION 6: RISK CLASSIFICATION
# ============================================================================

def classify_risk(percentage_score: float) -> str:
    """
    Classify risk level based on percentage score.
    """
    if percentage_score > 66:
        return '🔴 High Risk'
    elif percentage_score > 33:
        return '🟡 Medium Risk'
    else:
        return '🟢 Low Risk'


In [ ]:
# ============================================================================
# SECTION 7: DETAILED RISK BREAKDOWNS
# ============================================================================

def generate_risk_breakdown(
    selected_qubes: List[str],
    risk_matrix: pd.DataFrame,
    risk_weights: Dict[str, float]
) -> pd.DataFrame:
    """
    Generate detailed breakdown of risk contributions by DIMENSION.

    Shows which risk dimensions (Security, Compliance, etc.) contribute
    most to the total risk score for the selected iQube(s).

    Args:
        selected_qubes: List of iQube names to analyze
        risk_matrix: DataFrame with risk ratings from Book4
        risk_weights: Dictionary of dimension weights

    Returns:
        DataFrame with dimension contributions sorted by impact
    """
    # Convert to numeric scores
    risk_df = risk_matrix.replace(SCORE_MAPPING)
    risk_df = risk_df.infer_objects(copy=False).astype(float)

    dimension_scores = {}

    for dimension in risk_df.columns:
        total_score = 0
        weight = risk_weights.get(dimension, 1.0)

        for qube in selected_qubes:
            if qube in IQUBES_DICT:
                for data_type in IQUBES_DICT[qube]:
                    if data_type in risk_df.index:
                        try:
                            data_row = risk_df.loc[data_type]
                            if isinstance(data_row, pd.DataFrame):
                                score = data_row.iloc[0][dimension]
                            else:
                                score = data_row[dimension]
                            total_score += float(score) * weight
                        except:
                            continue

        dimension_scores[dimension] = total_score

    # Create DataFrame and sort by score
    breakdown_df = pd.DataFrame.from_dict(
        dimension_scores,
        orient='index',
        columns=['Weighted Score']
    )

    total = breakdown_df['Weighted Score'].sum()
    breakdown_df['Percentage'] = (
        breakdown_df['Weighted Score'] / total * 100
    ) if total > 0 else 0

    breakdown_df = breakdown_df.sort_values('Weighted Score', ascending=False)

    return breakdown_df


def generate_data_type_breakdown(
    selected_qubes: List[str],
    risk_matrix: pd.DataFrame,
    risk_weights: Dict[str, float]
) -> pd.DataFrame:
    """
    Generate breakdown showing which DATA TYPES contribute most to total risk.

    Shows which specific data fields (SSN, Credit Score, etc.) in the selected
    iQube(s) are driving the overall risk score.

    Args:
        selected_qubes: List of iQube names to analyze
        risk_matrix: DataFrame with risk ratings from Book4
        risk_weights: Dictionary of dimension weights

    Returns:
        DataFrame with data type contributions sorted by impact
    """
    # Convert to numeric scores
    risk_df = risk_matrix.replace(SCORE_MAPPING)
    risk_df = risk_df.infer_objects(copy=False).astype(float)

    data_type_scores = {}

    for qube in selected_qubes:
        if qube not in IQUBES_DICT:
            continue

        for data_type in IQUBES_DICT[qube]:
            if data_type not in risk_df.index:
                continue

            try:
                data_row = risk_df.loc[data_type]
                if isinstance(data_row, pd.DataFrame):
                    data_row = data_row.iloc[0]

                # Calculate total weighted contribution for this data type
                # across ALL dimensions
                total_contribution = 0
                for dimension in risk_df.columns:
                    weight = risk_weights.get(dimension, 1.0)
                    score = float(data_row[dimension])
                    total_contribution += score * weight

                data_type_scores[data_type] = total_contribution

            except Exception as e:
                continue

    # Create DataFrame
    breakdown_df = pd.DataFrame.from_dict(
        data_type_scores,
        orient='index',
        columns=['Total Contribution']
    )

    total = breakdown_df['Total Contribution'].sum()
    breakdown_df['Percentage'] = (
        breakdown_df['Total Contribution'] / total * 100
    ) if total > 0 else 0

    breakdown_df = breakdown_df.sort_values('Total Contribution', ascending=False)

    return breakdown_df


In [ ]:
# ============================================================================
# SECTION 8: USER INTERFACE
# ============================================================================

def get_user_input_qubes() -> List[str]:
    """
    Prompt user to select iQubes for analysis.
    """
    available_qubes = list(IQUBES_DICT.keys())

    print("\n" + "="*70)
    print("AVAILABLE IQUBES:")
    print("="*70)
    for idx, qube in enumerate(available_qubes, 1):
        print(f"{idx}. {qube}")
    print("="*70 + "\n")

    while True:
        user_input = input(
            "Enter iQube names separated by commas\n"
            "(e.g., 'Open Bank Account Qube, Car Finance Qube'): "
        )

        selected_qubes = [qube.strip() for qube in user_input.split(',')]
        invalid_qubes = [q for q in selected_qubes if q not in available_qubes]

        if invalid_qubes:
            print(f"\nInvalid iQubes: {invalid_qubes}")
            print("Please try again.\n")
        else:
            return selected_qubes


In [ ]:

# ============================================================================
# SECTION 9: MAIN EXECUTION FUNCTION
# ============================================================================

def main():
    """
    Main execution function - runs the complete risk assessment workflow.
    """
    global RISK_WEIGHTS  # Allow modification of global weights

    print("\n" + "="*70)
    print("           METAME DATA RISK ASSESSMENT SYSTEM")
    print("="*70 + "\n")

    # Simple file path for Google Colab (file should be uploaded to Colab)
    file_path = 'Book4.xlsx'
    sheet_name = 'Sheet8'

    # Load risk matrix from Book4
    print("Loading risk matrix from Book4...")
    try:
        risk_matrix = load_risk_matrix_from_book4(file_path, sheet_name)
        print(f"Successfully loaded {len(risk_matrix)} data types across {len(risk_matrix.columns)} risk dimensions\n")
    except FileNotFoundError:
        print("Error: Book4.xlsx not found!")
        print("Please upload Book4.xlsx using the file browser on the left")
        return
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    # DERIVE WEIGHTS EMPIRICALLY FROM BOOK4
    print("Deriving dimension weights from Book4 data...")
    RISK_WEIGHTS = derive_weights_from_book4(risk_matrix)
    print(f"Successfully calculated empirical weights for {len(RISK_WEIGHTS)} dimensions\n")

    # Display derived weights
    print("="*70)
    print("DERIVED DIMENSION WEIGHTS (Sorted by Importance)")
    print("="*70)
    sorted_weights = sorted(RISK_WEIGHTS.items(), key=lambda x: x[1], reverse=True)
    for i, (dim, weight) in enumerate(sorted_weights, 1):
        print(f"{i:2d}. {dim:25s}: {weight:.3f}")
    print("="*70 + "\n")

    # Display weight derivation details
    print_weight_derivation_details(risk_matrix, RISK_WEIGHTS)

    # Get user input for iQubes
    selected_qubes = get_user_input_qubes()

    # Calculate BASIC (unweighted) risk score
    print("\n" + "="*70)
    print("BASIC RISK ASSESSMENT (UNWEIGHTED)")
    print("="*70)
    basic_score, basic_pct, basic_class = calculate_basic_risk_score(
        selected_qubes, risk_matrix
    )
    print(f"Total Score: {basic_score}")
    print(f"Percentage: {basic_pct:.2f}%")
    print(f"Classification: {basic_class}")

    # Calculate WEIGHTED risk score
    print("\n" + "="*70)
    print("WEIGHTED RISK ASSESSMENT (EMPIRICALLY-DERIVED WEIGHTS)")
    print("="*70)
    weighted_score, weighted_pct, weighted_class = calculate_weighted_risk_score(
        selected_qubes, risk_matrix, RISK_WEIGHTS
    )
    print(f"Weighted Score: {weighted_score:.2f}")
    print(f"Percentage: {weighted_pct:.2f}%")
    print(f"Classification: {weighted_class}")

    # Generate dimension breakdown
    print("\n" + "="*70)
    print("RISK DIMENSION BREAKDOWN (TOP 10)")
    print("="*70)
    print("\nShows which risk dimensions contribute most to overall risk:\n")
    dimension_breakdown = generate_risk_breakdown(selected_qubes, risk_matrix, RISK_WEIGHTS)
    print(dimension_breakdown.head(10).to_string())

    # Generate data type breakdown
    print("\n" + "="*70)
    print("DATA TYPE CONTRIBUTION BREAKDOWN")
    print("="*70)
    print("\nShows which specific data fields drive the risk score:\n")
    data_type_breakdown = generate_data_type_breakdown(selected_qubes, risk_matrix, RISK_WEIGHTS)
    print(data_type_breakdown.to_string())

    # Summary
    print("\n" + "="*70)
    print("ASSESSMENT SUMMARY")
    print("="*70)
    print(f"\niQube(s) Analyzed: {', '.join(selected_qubes)}")
    print(f"Total Data Types: {len(data_type_breakdown)}")
    print(f"Risk Score: {weighted_pct:.2f}%")
    print(f"Classification: {weighted_class}")

    if len(data_type_breakdown) > 0:
        top_risk = data_type_breakdown.index[0]
        top_pct = data_type_breakdown.iloc[0]['Percentage']
        print(f"\nHighest Risk Data Type: {top_risk} ({top_pct:.1f}% of total risk)")

    print("\n" + "="*70)
    print("ASSESSMENT COMPLETE")
    print("="*70 + "\n")


In [ ]:
# ============================================================================
# SECTION 10: EXECUTE
# ============================================================================

if __name__ == "__main__":
    main()


           METAME DATA RISK ASSESSMENT SYSTEM

Loading risk matrix from Book4...
Successfully loaded 105 data types across 19 risk dimensions

Deriving dimension weights from Book4 data...
Successfully calculated empirical weights for 19 dimensions

DERIVED DIMENSION WEIGHTS (Sorted by Importance)
 1. Reputation               : 1.378
 2. Compliance               : 1.311
 3. Legal                    : 1.295
 4. Strategic                : 1.295
 5. Commercial               : 1.262
 6. Operational              : 1.251
 7. Security                 : 1.223
 8. Financial                : 1.201
 9. Competitiveness          : 1.174
10. Emotional                : 1.157
11. Intelligence             : 1.152
12. Social                   : 1.075
13. Health and Safety        : 0.667
14. Diplomatic               : 0.617
15. Political                : 0.595
16. Geopolitical             : 0.595
17. Military                 : 0.584
18. Public Welfare           : 0.584
19. Environmental            : 0.5

/tmp/ipython-input-3762699025.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  risk_df = risk_matrix.replace(SCORE_MAPPING).infer_objects(copy=False)
/tmp/ipython-input-2173006117.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  risk_df = risk_matrix.replace(SCORE_MAPPING)
/tmp/ipython-input-2173006117.py:87: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior,